# DMMR zero check

Two 6-hour runs on all eight modules, fixed range 0. Between runs: verified stop, range restoration, port closure, 5-minute pause, then reconnection. Allow **at least 12 h 5 min**. The DMMR stays powered.

1. Use Windows, 64-bit Python, Jupyter, NumPy, pandas and Matplotlib. Close Explorer and CGC software.
2. Check **COM15** and the settings below, then **Restart Kernel and Run All**. The driver is found in `../dmmr/` or an installed plugin folder; set `PLUGIN_DIR` if needed.
3. Keep the PC plugged in. Automatic sleep is inhibited; do not close the lid or trigger sleep manually.

Inputs default to open and unshielded. Record power-on time and conditions in `notes`. Use `open_shielded` only with shielding fitted. Before touching inputs, disconnect and verify external bias; shielding must not touch the centre contact.

An acquisition or shutdown failure cancels later runs. If shutdown is unconfirmed, check the instrument locally before restarting the kernel; never retry an open or blocked connection.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import collections
import contextlib
import ctypes
import csv
import hashlib
import importlib.util
import json
import math
import os
import struct
import sys
import time
import numpy as np
import pandas as pd

COM_PORT = 15
BAUD_RATE = 230400
PLUGIN_DIR = None
OUTPUT_ROOT = Path.cwd() / 'logs' / 'dmmr_zero'

PROTOCOL = {
    'input_condition': 'open_unshielded',  # Use 'open_shielded' only with shielding fitted.
    'ambient_temperature_C': None,        # Only if measured separately.
    'notes': '',                         # Power-on time, ventilation, etc.
    'range': 0,
    'series_count': 2,
    'duration_s': 6 * 60 * 60.0,          # Per series; 12 h of data in total.
    'pause_s': 5 * 60.0,                 # Port closed; DMMR stays powered.
    'diagnostics_interval_s': 60.0,
    'bin_s': 60.0,
    'baseline_window_s': 30 * 60.0,       # Non-overlapping analysis windows.
    'tail_s': 5 * 60.0,
    'stall_timeout_s': 30.0,              # Stop if any module stops sending valid readings.
    'io_timeout_s': 5.0,
}

## Factory reference

The 18 Feb 2025 CGC certificate reports 200 points per range with open, screened inputs. Cadence, warm-up time, temperature and acceptance limits are unspecified. "Mean Deviation" is undefined, not an offset tolerance or a known standard deviation. The [DMM-1PA noise figures](https://www.cgc-instruments.com/en/Products/Experiment-Control/DMMR/DMM-1PA) are not offset tolerances either.

Values below follow the printed order and are matched by P/N, not address. P/N 132310 had +534.305 fA in range 0. P/N 132303 has an ambiguous range order, so its difference from the certificate is not calculated. The suspect 1 µA table for P/N 132308 is unused. The PDF hash is retained with the data.


In [ ]:
# Visually checked transcription of pages 3-10, in printed order.
CERTIFICATE_SHA256 = '72a36960479b9c94c18138921a5e135538bdd670014b47824760d89a54614189'
CERTIFICATE_DATE = '2025-02-18'
# P/N: (means in A for ranges 0..4, "Mean Deviation" in A).
CERTIFICATE = {
    132303: ([-1.269036e-10, -3.490125e-12, -3.894800e-13, -1.974750e-13, -8.585000e-15],
             [2.4e-11, 2.5e-12, 2.4e-13, 3.4e-14, 1.1e-14]),
    132304: ([+5.731000e-14, -1.005750e-13, -9.767500e-14, -8.240765e-12, -1.291944e-10],
             [1.0e-14, 3.6e-14, 2.5e-13, 2.3e-12, 2.4e-11]),
    132305: ([-1.460000e-14, +1.758000e-14, +2.006405e-12, +1.635747e-11, -1.038038e-10],
             [1.1e-14, 3.7e-14, 2.5e-13, 2.5e-12, 2.4e-11]),
    132306: ([+1.476500e-14, +1.048750e-13, +1.387500e-14, +6.466600e-13, -7.309525e-11],
             [1.1e-14, 3.3e-14, 2.4e-13, 2.6e-12, 2.5e-11]),
    132307: ([+7.665000e-15, -1.090900e-13, -7.440000e-15, -4.662260e-12, -8.422875e-11],
             [1.0e-14, 3.4e-14, 2.6e-13, 2.3e-12, 2.3e-11]),
    132308: ([+1.185000e-14, +6.838500e-14, +7.376200e-13, +9.401575e-12, -1.959550e-11],
             [1.1e-14, 3.5e-14, 2.8e-13, 2.3e-12, 2.7e-11]),
    132309: ([+7.040000e-15, +1.646200e-13, +5.774650e-13, +9.554995e-12, -2.145025e-11],
             [1.1e-14, 3.6e-14, 2.7e-13, 2.2e-12, 2.4e-11]),
    132310: ([+5.343050e-13, +6.315450e-13, +2.713625e-12, +1.637357e-11, -4.442775e-11],
             [1.4e-14, 3.6e-14, 2.4e-13, 2.5e-12, 2.6e-11]),
}
AMBIGUOUS_CERTIFICATE_PNS = {132303}
certificate_table = pd.DataFrame([
    {'P/N': pn, 'pdf_page': pn - 132300, 'range': r,
     'offset_CGC_pA': means[r] * 1e12,
     'Mean_Deviation_CGC_fA': deviations[r] * 1e15,
     'order_unconfirmed': pn in AMBIGUOUS_CERTIFICATE_PNS}
    for pn, (means, deviations) in CERTIFICATE.items() for r in range(5)
])

## Recording

Raw frames include startup transients. Repeated/decreasing timestamps within a segment are excluded from statistics; equal currents with new timestamps are valid. No corrections or calibration changes.

Minute diagnostics: controller/chassis and module temperatures, voltages, fan, states and uptime. Read times are saved; sensor conversion times are unavailable. Commands wait for an empty current queue. No diagnostics during pauses.

Receive errors −10 to −13 are logged. Initial range reads allow one checked retry. Range setup writes only changed settings and allows one verified recovery, with at most one purge; writes are never replayed.

In-run recovery verifies controller, mode and ranges, with one purge/restart at most. Lost data stay missing; purges start new timestamp segments. Stop after a second fault before all modules reply, or a fourth incident within 60 s.

Cleanup has one separate resynchronization attempt. Reconnection requires verified range restoration, both OFF states and port closure. Never retry or close a blocked DLL.


In [ ]:
def explorer_plugin_dirs():
    '''ESIBD Explorer's configured plugin folder (Windows settings), then its default.'''
    found = []
    try:
        import winreg
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Software\ESIBD LAB\ESIBD Explorer\General') as key:
            value = winreg.QueryValueEx(key, 'Plugin path')[0]
        if isinstance(value, str) and value and not value.startswith('@'):
            found.append(Path(value).expanduser())
    except (ImportError, OSError):
        pass
    found.append(Path.home() / 'ESIBD Explorer' / 'plugins')
    return found


def find_plugin_dir(plugin_dir=None):
    """Find the DMMR plugin from the repository, notebook or installed layout."""
    if plugin_dir is not None:
        return Path(plugin_dir).expanduser().resolve()
    cwd = Path.cwd()
    candidates = [cwd, cwd / 'dmmr', cwd.parent / 'dmmr'] + [plugins / 'dmmr' for plugins in explorer_plugin_dirs()]
    for candidate in candidates:
        if (candidate / 'dmmr_plugin.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('DMMR plugin not found; set PLUGIN_DIR in the settings.')


def load_driver(plugin_dir):
    """Load only the runtime bundled with this DMMR plugin."""
    root = Path(plugin_dir).resolve() / 'vendor' / 'runtime'
    name = '_esibd_bundled_dmmr_zero_check'
    if not (root / '__init__.py').is_file():
        raise ModuleNotFoundError(f'DMMR runtime missing: {root}')
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != root:
        raise RuntimeError('Another DMMR runtime is loaded; restart the kernel.')
    if name not in sys.modules:
        spec = importlib.util.spec_from_file_location(
            name, root / '__init__.py', submodule_search_locations=[str(root)])
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            spec.loader.exec_module(module)
        except BaseException:
            for key in list(sys.modules):
                if key == name or key.startswith(name + '.'):
                    del sys.modules[key]
            raise
    return sys.modules[name].DMMR


def checked(device, action, result):
    status, values = (result[0], result[1:]) if isinstance(result, tuple) else (result, ())
    if status != device.NO_ERR:
        raise RuntimeError(f'{action}: status {status}')
    return values


def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [clean_json(v) for v in value]
    if isinstance(value, np.generic):
        return clean_json(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def save_report(output_dir, report):
    path = Path(output_dir) / 'report.json'
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(clean_json(report), indent=2, ensure_ascii=False,
                                    allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def stop_stream(device, timeout):
    checked(device, 'automatic OFF', device.set_automatic_current(False, timeout_s=timeout))
    (enabled,) = checked(device, 'automatic OFF readback', device.get_automatic_current(timeout_s=timeout))
    if enabled is not False:
        raise RuntimeError('Automatic streaming is still enabled.')


def verify_controller(device, timeout):
    state, label = checked(device, 'controller state', device.get_state(timeout_s=timeout))
    value = int(state, 16) if isinstance(state, str) else int(state)
    if value != 0:  # COM_DMMR_8_ST_ON = 0 in the CGC header.
        raise RuntimeError(f'Unexpected controller state: {state} ({label}).')


def read_range(device, address, timeout, events, action):
    """Retry a receive error once, separated by a verified controller reply.

    Keep timeouts unchanged; never retry a blocked transport.
    """
    for attempt in (1, 2):
        result = device.get_module_meas_range(address, timeout_s=timeout)
        status = result[0]
        events.append({'action': action, 'address': address, 'attempt': attempt,
                       'status': status, 'utc': datetime.now(timezone.utc).isoformat(),
                       'range': result[1] if status == device.NO_ERR else None,
                       'auto': result[2] if status == device.NO_ERR else None})
        if status in (-10, -11, -12, -13) and attempt == 1 and not getattr(device, '_transport_poisoned', False):
            print(f'{action}, module {address}: {status}; checking controller before one retry.', flush=True)
            verify_controller(device, timeout)
            continue
        actual, auto = checked(device, f'{action}, module {address}, read {attempt}', result)
        if type(actual) is not int or actual not in range(5) or type(auto) is not bool:
            raise RuntimeError(f'{action}, module {address}: invalid range response ({actual}, {auto}).')
        return actual, auto


def get_stream_frame(device, timeout):
    # GetCurent can hide errors from the DLL automatic-input parser.
    parsed = device._call_locked_with_timeout(device.check_auto_input, timeout, 'check_auto_input')
    if parsed[0] != device.NO_ERR:
        return parsed[0], None, math.nan, None, math.nan
    return device.get_current(timeout_s=timeout)


def flush_frames(device, timeout, limit=10000):
    # Flush only after stopping the stream, never during acquisition.
    for count in range(limit):
        result = get_stream_frame(device, timeout)
        if result[0] == device.NO_DATA:
            return count
        checked(device, 'purge FIFO', result)
    raise RuntimeError('Current FIFO did not empty after stopping the stream.')


def configure_range(device, addresses, requested, timeout, events, changed, recoveries=None):
    """Write only changed settings; verify a lost ACK without replaying the write."""
    recovery = device.new_command_recovery(phase='range_setup', timeout_s=timeout)
    verified = {}

    def validate(address, values, expected_auto=None, fixed=None):
        actual, auto = values
        if (type(actual) is not int or actual not in range(5) or type(auto) is not bool
                or (expected_auto is not None and auto is not expected_auto)
                or (fixed is not None and actual != fixed)):
            raise RuntimeError(f'Module {address}: range not confirmed ({actual}, auto={auto}); '
                               f'expected auto={expected_auto}, range={fixed}.')
        return actual, auto

    def log_read(address, result):
        status = result[0]
        events.append({'action': 'range readback', 'address': address,
                       'status': status, 'utc': datetime.now(timezone.utc).isoformat(),
                       'range': result[1] if status == device.NO_ERR else None,
                       'auto': result[2] if status == device.NO_ERR else None})

    def recover(status, action, address, auto=None, fixed=None):
        def verify():
            recovery.verify_running()
            for previous, value in verified.items():
                validate(previous, recovery._call('get_module_meas_range', previous), False, value)
            result = recovery._call('get_module_meas_range', address)
            log_read(address, (device.NO_ERR, *result))
            return validate(address, result, auto, fixed)
        print(f'{action}, module {address}: {status}; verifying hardware without replay.', flush=True)
        try:
            return recovery.recover(status, f'{action}, module {address}', verify)
        finally:
            if recoveries is not None and recovery.last_incident is not None:
                recoveries.append(dict(recovery.last_incident))

    def readback(address, auto=None, fixed=None):
        recovery._active()
        result = device.get_module_meas_range(address, timeout_s=timeout)
        recovery._active()
        log_read(address, result)
        if result[0] != device.NO_ERR:
            return recover(result[0], 'range readback', address, auto, fixed)
        return validate(address, result[1:], auto, fixed)

    def write(address, method, value, auto, fixed=None):
        recovery._active()
        changed.add(address)  # A lost ACK may still mean the hardware changed.
        status = getattr(device, method)(address, value, timeout_s=timeout)
        recovery._active()
        if status != device.NO_ERR:
            return recover(status, method, address, auto, fixed)
        return readback(address, auto, fixed)

    for address in addresses:
        actual, auto = readback(address)
        if auto:
            actual, auto = write(address, 'set_module_auto_range', False, False)
        if actual != requested:
            actual, auto = write(address, 'set_module_meas_range', requested, False, requested)
        validate(address, (actual, auto), False, requested)
        verified[address] = requested


def finish_capture(device, output_dir, report, timeout, key='diagnostics', filename='native_startup.log'):
    diagnostic = report[key]
    if diagnostic.get('ended'):
        return
    diagnostic['ended'] = True
    if getattr(device, '_transport_poisoned', False):
        text = 'DLL blocked; capture left open. No concurrent calls. See the partial log path printed at startup.'
    else:
        try:
            text = device.end_startup_diagnostics(timeout_s=timeout)
        except Exception as exc:
            text = f'Native capture unavailable: {type(exc).__name__}: {exc}'
    try:
        (Path(output_dir) / filename).write_text(
            diagnostic.get('start', '') + '\n' + text + '\n', encoding='utf-8')
        diagnostic['file'] = filename
    except OSError as exc:
        # Optional native evidence must never prevent hardware shutdown.
        diagnostic['save_error'] = f'{type(exc).__name__}: {exc}'
        print(f'WARNING: could not save {filename}: {exc}', flush=True)


RAW_COLUMNS = ['phase', 'address', 'product_no', 'requested_range', 'actual_range',
               'host_utc', 'host_elapsed_s', 'device_time_s', 'current_A',
               'status', 'selected', 'reason', 'segment']


def collect_run(device, modules, cfg, writer, raw_file, report, output_dir,
                clock=time.monotonic, sleep=time.sleep):
    """Record for the full duration, including startup, without a point limit."""
    start = clock()
    counts = {a: 0 for a in modules}
    last_time, last_valid = {}, {a: start for a in modules}
    reasons = collections.Counter()
    info = {'points_by_address': counts, 'duration_s': 0.0, 'empty_polls': 0,
            'started_monotonic_s': start, 'frame_reasons': {}, 'complete': False}
    report['acquisition'] = info
    next_progress = cfg['bin_s']
    telemetry_file = (Path(output_dir) / 'telemetry.csv').open('w', newline='', encoding='utf-8')
    recovery = device.new_read_recovery({a: (cfg['range'], False) for a in modules},
                                        automatic=True, timeout_s=cfg['io_timeout_s'], clock=clock)
    info['read_recoveries'] = []
    read_grace = start
    segment = 0

    def recover_read(status, action):
        nonlocal read_grace, segment
        incident_start = clock() - start
        try:
            incident = recovery.recover(status, action)
            if incident['purged']:
                # The old FIFO is confirmed empty. Device time may restart;
                # retain its raw values, but never compare across this boundary.
                segment += 1
                last_time.clear()
            incident['segment_after'] = segment
            # A bounded grace period, not a fictitious sample or timestamp.
            read_grace = clock()
        finally:
            incident = recovery.last_incident
            if incident is not None:
                incident.update(start_elapsed_s=incident_start, finished_elapsed_s=clock() - start)
                info['read_recoveries'].append(incident)
            save_report(output_dir, report)

    diagnostics = DiagnosticRecorder(device, modules, cfg, telemetry_file, report, start, clock,
                                    recover_read=recover_read)
    try:
        checked(device, 'automatic ON', device.set_automatic_current(True, timeout_s=cfg['io_timeout_s']))
        # Capture the startup ACK, not the continuous stream.
        finish_capture(device, output_dir, report, cfg['io_timeout_s'])
        if getattr(device, '_transport_poisoned', False):
            raise RuntimeError('DLL blocked during native capture.')
        while clock() - start < cfg['duration_s']:
            stale = [a for a in modules if clock() - max(last_valid[a], read_grace) >= cfg['stall_timeout_s']]
            if stale:
                raise TimeoutError(f'No new valid reading for {cfg["stall_timeout_s"]:g} s: modules {stale}.')
            status, address, current, actual, device_time = get_stream_frame(device, cfg['io_timeout_s'])
            now = clock()
            if status == device.NO_DATA:
                info['empty_polls'] += 1
                # Read one diagnostic with an empty FIFO; the DLL buffers incoming currents.
                diagnostics.poll_one()
                sleep(0.01)
                continue
            reason, fatal = 'selected', False
            if status != device.NO_ERR:
                reason, fatal = f'status_{status}', True
            elif address not in modules:
                reason, fatal = 'unknown_address', True
            elif not (math.isfinite(current) and math.isfinite(device_time)):
                reason, fatal = 'nonfinite', True
            elif type(actual) is not int or actual not in range(5):
                reason, fatal = 'invalid_range', True
            elif actual != cfg['range']:
                reason, fatal = 'wrong_range', True
            elif device_time <= last_time.get(address, -math.inf):
                reason = 'timestamp_not_increasing'
            elif now - start >= cfg['duration_s']:
                reason = 'after_duration'
            else:
                last_time[address] = device_time
                last_valid[address] = now
                counts[address] += 1
                awaiting = bool(recovery.awaiting_samples)
                recovery.note_sample(address)
                if awaiting and not recovery.awaiting_samples:
                    recovery.last_incident['resumed_elapsed_s'] = now - start
            selected = reason == 'selected'
            reasons[reason] += 1
            valid_payload = status == device.NO_ERR
            writer.writerow({
                'phase': 'continuous', 'address': address if valid_payload else '',
                'product_no': modules.get(address, {}).get('product_no', '') if valid_payload else '',
                'requested_range': cfg['range'], 'actual_range': actual if valid_payload else '',
                'host_utc': datetime.now(timezone.utc).isoformat(), 'host_elapsed_s': now - start,
                'device_time_s': device_time if valid_payload else '', 'current_A': current if valid_payload else '',
                'status': status, 'selected': int(selected), 'reason': reason, 'segment': segment})
            raw_file.flush()
            if fatal:
                if status in (-10, -11, -12, -13):
                    recover_read(status, 'current stream')
                    continue
                raise RuntimeError(f'Acquisition, module {address}: {reason}')
            if now - start >= next_progress:
                info['duration_s'] = now - start
                info['frame_reasons'] = dict(reasons)
                save_report(output_dir, report)
                print(f'{(now-start)/60:.1f}/{cfg["duration_s"]/60:g} min; points per module: {counts}', flush=True)
                next_progress += cfg['bin_s']
        stale = [a for a in modules if counts[a] < 2 or clock() - last_valid[a] >= cfg['stall_timeout_s']]
        if stale or recovery.awaiting_samples:
            raise TimeoutError(f'Incomplete acquisition for modules {stale or sorted(recovery.awaiting_samples)}.')
        info['complete'] = True
    finally:
        info['duration_s'] = clock() - start
        info['frame_reasons'] = dict(reasons)
        telemetry_file.close()


class CleanupCommands:
    """One port-buffer recovery for returned receive errors, only after recording."""

    def __init__(self, device, timeout, report):
        self.device, self.timeout, self.report = device, timeout, report
        report['operations'] = []
        self.recovery = report['recovery'] = {'attempted': False, 'succeeded': False}

    def call(self, action, method, *args):
        if getattr(self.device, '_transport_poisoned', False):
            raise RuntimeError('DLL blocked; no further calls.')
        record = {'action': action, 'arguments': args, 'utc': datetime.now(timezone.utc).isoformat()}
        self.report['operations'].append(record)
        try:
            result = method(*args, timeout_s=self.timeout)
        except (Exception, KeyboardInterrupt) as exc:
            record['error'] = f'{type(exc).__name__}: {exc}'
            raise
        status = result[0] if isinstance(result, tuple) else result
        record['status'] = status
        if isinstance(result, tuple):
            record['values'] = result[1:]
        if (status in (-10, -11, -12, -13) and not self.recovery['attempted']
                and not getattr(self.device, '_transport_poisoned', False)):
            self.recovery.update(attempted=True, action=action, first_status=status)
            print(f'Cleanup: {action} returned {status}; one communication resynchronization.', flush=True)
            try:
                # CGC Purge clears communication buffers, not calibration or saved CSVs.
                # It also returns the serial link to 9600 baud; the next series
                # reconnects and sets its requested baud rate before recording.
                # ReadRecovery handles in-run purges separately, including
                # baud restoration and an explicit boundary in the saved data.
                purge_status = self.device._call_locked_with_timeout(
                    self.device.purge, self.timeout, 'cleanup_purge')
                self.recovery['purge_status'] = purge_status
                checked(self.device, 'cleanup purge', purge_status)
                response = self.device.get_state(timeout_s=self.timeout)
                self.recovery['controller_reply'] = response
                state, label = checked(self.device, 'cleanup controller readback', response)
                if (int(state, 16) if isinstance(state, str) else int(state)) != 0:
                    raise RuntimeError(f'Unexpected controller state after purge: {state} ({label}).')
                values = self.call(action, method, *args)  # Global budget already consumed.
                self.recovery['succeeded'] = True
                return values
            except (Exception, KeyboardInterrupt) as exc:
                self.recovery['error'] = f'{type(exc).__name__}: {exc}'
                raise
        return checked(self.device, action, result)


def restore_and_shutdown(device, original, timeout, events=None):
    result = {'range_restoration_needed': bool(original), 'ranges_restored': False,
              'shutdown_confirmed': False, 'off_confirmed': False, 'errors': []}
    events = [] if events is None else events
    commands = CleanupCommands(device, timeout, result)
    if getattr(device, '_transport_poisoned', False):
        result['errors'].append('DLL blocked; no further calls. Shutdown unconfirmed. Check the instrument locally before restarting the kernel.')
        return result
    try:
        commands.call('automatic OFF', device.set_automatic_current, False)
        (enabled,) = commands.call('automatic OFF readback', device.get_automatic_current)
        if enabled is not False:
            raise RuntimeError('Automatic streaming is still enabled.')
        state, label = commands.call('controller state', device.get_state)
        if (int(state, 16) if isinstance(state, str) else int(state)) != 0:
            raise RuntimeError(f'Unexpected controller state at cleanup: {state} ({label}).')
        # Keep modules enabled until their ranges are restored. Auto mode does not
        # require forcing its previously observed, possibly obsolete fixed range.
        for address, (meas_range, auto) in original.items():
            commands.call(f'restore auto, module {address}', device.set_module_auto_range, address, auto)
            if not auto:
                commands.call(f'restore range, module {address}', device.set_module_meas_range, address, meas_range)
            actual, actual_auto = commands.call(f'restore readback, module {address}', device.get_module_meas_range, address)
            events.append({'action': 'restore readback', 'address': address, 'status': device.NO_ERR,
                           'range': actual, 'auto': actual_auto, 'utc': datetime.now(timezone.utc).isoformat()})
            if (type(actual) is not int or actual not in range(5) or type(actual_auto) is not bool
                    or actual_auto != auto or (not auto and actual != meas_range)):
                raise RuntimeError(f'Range restoration not confirmed for module {address}.')
        result['ranges_restored'] = True
    except (Exception, KeyboardInterrupt) as exc:
        result['errors'].append(f'Restoration: {type(exc).__name__}: {exc}')

    # Same OFF contract as the runtime: attempt both disables, require both fresh
    # NO_ERR/False readbacks, then confirm port closure. Never close after a failure.
    gates = [('automatic current', device.set_automatic_current, device.get_automatic_current),
             ('measurement enable', device.set_enable, device.get_enable)]
    shutdown_errors = []
    for operation in ('disable', 'verify'):
        for label, setter, getter in gates:
            if getattr(device, '_transport_poisoned', False):
                break
            try:
                if operation == 'disable':
                    commands.call('disable ' + label, setter, False)
                else:
                    (enabled,) = commands.call('verify ' + label, getter)
                    if enabled is not False:
                        raise RuntimeError(f'{label}: OFF not confirmed.')
            except (Exception, KeyboardInterrupt) as exc:
                shutdown_errors.append(f'{operation} {label}: {type(exc).__name__}: {exc}')
    if getattr(device, '_transport_poisoned', False):
        shutdown_errors.append('DLL blocked; no concurrent close call.')
    if not shutdown_errors:
        result['off_confirmed'] = True
        try:
            if device.disconnect() is not True or getattr(device, 'connected', False) or getattr(device, '_transport_poisoned', False):
                raise RuntimeError('Port closure not confirmed.')
            result['shutdown_confirmed'] = True
        except (Exception, KeyboardInterrupt) as exc:
            shutdown_errors.append(f'Disconnect: {type(exc).__name__}: {exc}')
    result['errors'].extend(shutdown_errors)
    return result


def validate_protocol(cfg):
    if cfg['input_condition'] not in ('open_unshielded', 'open_shielded'):
        raise ValueError('input_condition must be open_unshielded or open_shielded.')
    if isinstance(cfg['range'], bool) or not isinstance(cfg['range'], int) or cfg['range'] not in range(5):
        raise ValueError('range must be an integer from 0 to 4.')
    if type(cfg['series_count']) is not int or cfg['series_count'] < 1:
        raise ValueError('series_count must be a positive integer.')
    for key in ('duration_s', 'bin_s', 'tail_s', 'stall_timeout_s', 'io_timeout_s',
                'pause_s', 'diagnostics_interval_s'):
        value = cfg[key]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
            raise ValueError(f'{key} must be a finite positive number.')
    window = cfg.get('baseline_window_s', 1800.0)
    if isinstance(window, bool) or not isinstance(window, (int, float)) or not math.isfinite(window) or window <= 0:
        raise ValueError('baseline_window_s must be a finite positive number.')
    if any(cfg[key] > cfg['duration_s'] for key in ('bin_s', 'tail_s', 'stall_timeout_s')):
        raise ValueError('Analysis windows and stall timeout must not exceed acquisition duration.')


def run_zero_check(device, output_dir, cfg, provenance=None, clock=time.monotonic, sleep=time.sleep):
    validate_protocol(cfg)
    if getattr(device, '_transport_poisoned', False) or getattr(device, 'connected', False):
        raise RuntimeError('Use a disconnected, unblocked device; runs are not retried automatically.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    report = {'schema': 2, 'started_utc': datetime.now(timezone.utc).isoformat(),
              'protocol': dict(cfg), 'provenance': provenance or {}, 'modules': {},
              'certificate_sha256': CERTIFICATE_SHA256, 'certificate_date': CERTIFICATE_DATE,
              'certificate_values': CERTIFICATE, 'certificate_ambiguous_pns': sorted(AMBIGUOUS_CERTIFICATE_PNS),
              'conformity': 'not_assessable_no_acceptance_limits', 'status': 'running',
              'initial_ranges': {}, 'range_reads': [], 'range_recoveries': [], 'diagnostics': {},
              'acquisition': None, 'error': None, 'cleanup': None}
    original, changed = {}, set()
    connection_attempted = False
    save_report(output_dir, report)
    configure_log = getattr(device, 'configure_protocol_log', None)
    if callable(configure_log):
        configure_log(output_dir)
    try:
        with (output_dir / 'raw.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=RAW_COLUMNS)
            writer.writeheader()
            stream.flush()
            try:
                report['diagnostics']['start'] = device.begin_startup_diagnostics(timeout_s=cfg['io_timeout_s'])
                connection_attempted = True
                if device.connect(timeout_s=cfg['io_timeout_s']) is not True:
                    raise RuntimeError('Connection not confirmed.')
                stop_stream(device, cfg['io_timeout_s'])
                checked(device, 'enable', device.set_enable(True, timeout_s=cfg['io_timeout_s']))
                (enabled,) = checked(device, 'enable readback', device.get_enable(timeout_s=cfg['io_timeout_s']))
                if enabled is not True:
                    raise RuntimeError('Module enable not confirmed.')
                modules = device.initialize(timeout_s=cfg['io_timeout_s'], persist_scan=False)
                report['modules'] = modules
                pns = [info.get('product_no') for info in modules.values()]
                if len(modules) != 8 or len(set(pns)) != 8 or set(pns) != set(CERTIFICATE):
                    raise RuntimeError(f'Module identities do not match the certificate: {pns}. Do not match by position.')
                report['controller'] = device.get_product_info(timeout_s=cfg['io_timeout_s'])
                verify_controller(device, cfg['io_timeout_s'])
                for address in modules:
                    original[address] = read_range(device, address, cfg['io_timeout_s'], report['range_reads'], 'initial range')
                    report['initial_ranges'][address] = original[address]
                # Read all initial ranges before changing any of them.
                configure_range(device, modules, cfg['range'], cfg['io_timeout_s'], report['range_reads'], changed,
                                report['range_recoveries'])
                verify_controller(device, cfg['io_timeout_s'])
                report['frames_flushed_before_run'] = flush_frames(device, cfg['io_timeout_s'])
                save_report(output_dir, report)
                print(f'Recording: {cfg["duration_s"]/60:g} min, fixed range {cfg["range"]}, eight modules.', flush=True)
                collect_run(device, modules, cfg, writer, stream, report, output_dir, clock=clock, sleep=sleep)
                report['status'] = 'acquisition_complete'
            except (Exception, KeyboardInterrupt) as exc:
                report['status'] = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
                report['error'] = f'{type(exc).__name__}: {exc}'
                print(f'INCOMPLETE RUN: {report["error"]}', flush=True)
            finally:
                try:
                    finish_capture(device, output_dir, report, cfg['io_timeout_s'])
                except (Exception, KeyboardInterrupt) as exc:
                    report['diagnostics']['error'] = f'{type(exc).__name__}: {exc}'
                    if isinstance(exc, KeyboardInterrupt):
                        report.update(status='interrupted', error=report['error'] or report['diagnostics']['error'])
                if connection_attempted and (getattr(device, 'connected', False) or getattr(device, '_transport_poisoned', False)):
                    if not getattr(device, '_transport_poisoned', False):
                        capture = report['cleanup_diagnostics'] = {'start': ''}
                        try:
                            capture['start'] = device.begin_startup_diagnostics(timeout_s=cfg['io_timeout_s'])
                        except (Exception, KeyboardInterrupt) as exc:
                            capture['error'] = f'{type(exc).__name__}: {exc}'
                            if isinstance(exc, KeyboardInterrupt):
                                report.update(status='interrupted', error=report['error'] or capture['error'])
                    try:
                        report['cleanup'] = restore_and_shutdown(device, {a: original[a] for a in changed},
                                                                 cfg['io_timeout_s'], report['range_reads'])
                    finally:
                        if 'cleanup_diagnostics' in report:
                            finish_capture(device, output_dir, report, cfg['io_timeout_s'],
                                           key='cleanup_diagnostics', filename='native_cleanup.log')
                else:
                    report['cleanup'] = {'ranges_restored': not changed, 'shutdown_confirmed': False,
                                         'errors': ['No connection available to confirm shutdown.']}
    finally:
        report['finished_utc'] = datetime.now(timezone.utc).isoformat()
        report['protocol_diagnostics'] = getattr(device, 'protocol_diagnostics', lambda: {})()
        if getattr(device, '_transport_poisoned', False) and (report.get('cleanup') or {}).get('shutdown_confirmed'):
            report['cleanup']['errors'].append('DLL blocked after port closure; no reconnect permitted.')
        if report['status'] == 'acquisition_complete':
            report['status'] = 'complete' if report['cleanup'] and not report['cleanup']['errors'] else 'cleanup_failed'
        save_report(output_dir, report)
    print(f'Result: {report["status"]}; {output_dir}')
    if report['cleanup']['errors']:
        print('WARNING:', '\n'.join(report['cleanup']['errors']))
    return report


def sample_statistics(group, report):
    pn, actual = int(group['product_no'].iloc[0]), int(group['actual_range'].iloc[0])
    x = group['current_A'].to_numpy(dtype=float)
    multiple_segments = 'segment' in group and group['segment'].nunique() > 1
    time_source = 'host_elapsed_s' if multiple_segments else 'device_time_s'
    t = group[time_source].to_numpy(dtype=float)
    mean = float(x.mean())
    reference = report['certificate_values'][str(pn)][0][actual]
    ambiguous = pn in report['certificate_ambiguous_pns']
    return {
        'address': int(group['address'].iloc[0]), 'P/N': pn, 'range': actual, 'N': len(x),
        'mean_pA': mean * 1e12,
        'std_fA': float(np.std(x, ddof=1)) * 1e15 if len(x) > 1 else None,
        'mean_abs_deviation_fA': float(np.mean(np.abs(x - mean))) * 1e15,
        'peak_to_peak_fA': float(np.ptp(x)) * 1e15,
        'drift_fA_min': float(np.polyfit(t - t[0], x - mean, 1)[0]) * 60e15 if not multiple_segments and len(t) > 1 and np.ptp(t) > 0 else None,
        'observed_duration_s': float(np.ptp(t)),
        'time_source': time_source,
        'host_start_s': float(group['host_elapsed_s'].min()),
        'host_end_s': float(group['host_elapsed_s'].max()),
        'median_step_s': float(np.median(np.diff(t))) if len(t) > 1 else None,
        'max_step_s': float(np.max(np.diff(t))) if len(t) > 1 else None,
        'offset_CGC_pA': reference * 1e12,
        'Mean_Deviation_CGC_fA': report['certificate_values'][str(pn)][1][actual] * 1e15,
        'certificate_difference_pA': None if ambiguous else (mean - reference) * 1e12,
        'comparison': 'CGC range order unconfirmed' if ambiguous else ('unshielded input' if report['protocol']['input_condition'] == 'open_unshielded' else 'CGC acceptance limits not provided'),
    }


def analyse_run(output_dir):
    output_dir = Path(output_dir)
    report = json.loads((output_dir / 'report.json').read_text(encoding='utf-8'))
    if report.get('schema') != 2:
        raise ValueError('Unsupported protocol: range-sweep phases cannot be analysed as a continuous run.')
    raw = pd.read_csv(output_dir / 'raw.csv', float_precision='round_trip')
    samples = raw.loc[(raw['selected'] == 1) & (raw['status'] == 0)].copy()
    summary, binned, tail, windows = [], [], [], []
    cfg = report['protocol']
    window_s = cfg.get('baseline_window_s', 1800.0)
    acquired_s = min((report.get('acquisition') or {}).get('duration_s', 0.0), cfg['duration_s'])
    recoveries = (report.get('acquisition') or {}).get('read_recoveries', [])
    for _, group in samples.groupby(['address', 'product_no', 'actual_range'], sort=True):
        summary.append(sample_statistics(group, report))
        bins = np.floor(group['host_elapsed_s'] / cfg['bin_s']).astype(int)
        for index, chunk in group.groupby(bins):
            row = sample_statistics(chunk, report)
            row.update(start_s=float(index * cfg['bin_s']),
                       end_s=float(min((index + 1) * cfg['bin_s'], cfg['duration_s'])))
            binned.append(row)
        # No drift fit across a stream restart, including a communication purge.
        for index, chunk in group.groupby(np.floor(group['host_elapsed_s'] / window_s).astype(int)):
            end = float((index + 1) * window_s)
            crosses_restart = any(r.get('purged') and r['start_elapsed_s'] < end
                                  and r.get('resumed_elapsed_s', r['finished_elapsed_s']) > index * window_s
                                  for r in recoveries)
            if end <= acquired_s and not crosses_restart:
                row = sample_statistics(chunk, report)
                row.update(start_s=float(index * window_s), end_s=end)
                windows.append(row)
        # Final-window statistics only for fully acquired runs.
        if (report.get('acquisition') or {}).get('complete'):
            chunk = group.loc[group['host_elapsed_s'] >= cfg['duration_s'] - cfg['tail_s']]
            if len(chunk):
                row = sample_statistics(chunk, report)
                row['stability'] = 'review the trace and minute means'
                tail.append(row)
    report['summary'] = summary
    report['time_bins'] = binned
    report['tail_summary'] = tail
    report['baseline_windows'] = windows
    report['frame_reasons'] = raw['reason'].value_counts().to_dict()
    save_report(output_dir, report)
    return raw, pd.DataFrame(summary), report



# Diagnostics are read-only: no manual current reads or calibration, fan or bias changes.
MODULE_HK_FIELDS = [
    'volt_3v3_v', 'temp_cpu_c', 'volt_5v0_v', 'volt_12v_v',
    'volt_3v3i_v', 'temp_cpui_c', 'volt_2v5i_v', 'volt_36vn_v',
    'volt_20vp_v', 'volt_20vn_v', 'volt_15vp_v', 'volt_15vn_v',
    'volt_1v8p_v', 'volt_1v8n_v', 'volt_vrefp_v', 'volt_vrefn_v',
]
TELEMETRY_COLUMNS = ['host_utc', 'host_elapsed_s', 'read_duration_s', 'scope',
                     'address', 'product_no', 'command', 'field', 'value', 'status', 'error']


def diagnostic_commands(modules):
    """Field order and units follow the CGC COM-DMMR-8.h header."""
    state_fields = ['state_hex', 'flags']
    uptime_fields = ['uptime_s', 'uptime_ms', 'total_uptime_s', 'total_uptime_ms']
    commands = [
        ('controller', None, 'get_state', ['state_hex', 'state_name']),
        ('controller', None, 'get_housekeeping', ['volt_12v_v', 'volt_5v0_v', 'volt_3v3_v', 'temp_cpu_c']),
        ('base', None, 'get_base_temp', ['temperature_c']),
        ('base', None, 'get_base_fan_rpm', ['fan_rpm']),
        ('base', None, 'get_base_fan_pwm', ['fan_pwm', 'state_hex', 'flags']),
        ('controller', None, 'get_device_state', state_fields),
        ('controller', None, 'get_voltage_state', state_fields),
        ('controller', None, 'get_temperature_state', state_fields),
        ('controller', None, 'get_uptime_int', uptime_fields),
    ]
    for address in sorted(modules):
        commands.extend([
            ('module', address, 'get_module_housekeeping', MODULE_HK_FIELDS),
            ('module', address, 'get_module_state', ['state']),
            ('module', address, 'get_module_uptime_int', uptime_fields),
        ])
    return commands


class DiagnosticRecorder:
    """Read one diagnostic, then drain the current FIFO.

    Log read times and errors; the API exposes no sensor conversion timestamp.
    """
    def __init__(self, device, modules, cfg, stream, report, start, clock, recover_read=None):
        self.device, self.modules, self.cfg = device, modules, cfg
        self.recover_read = recover_read
        self.stream, self.report, self.start, self.clock = stream, report, start, clock
        self.writer = csv.DictWriter(stream, fieldnames=TELEMETRY_COLUMNS)
        self.writer.writeheader()
        stream.flush()
        # Use raw DMMRBase getters: public wrappers would acquire the lock twice.
        self.backend = getattr(device, '_backend', device)
        self.base = next((cls for cls in type(self.backend).__mro__ if cls.__name__ == 'DMMRBase'), None)
        self.commands = diagnostic_commands(modules)
        self.pending = collections.deque()
        self.next_cycle = 0.0
        self.info = report['telemetry'] = {'file': 'telemetry.csv', 'calls': 0,
            'failed_calls': 0, 'invalid_fields': 0, 'cycles_started': 0, 'cycles_completed': 0,
            'conversion_freshness': 'not_exposed_by_housekeeping_API'}

    def poll_one(self):
        elapsed = self.clock() - self.start
        if elapsed >= self.cfg['duration_s'] - self.cfg['io_timeout_s']:
            return
        if not self.pending:
            if elapsed < self.next_cycle:
                return
            self.pending.extend(self.commands)
            # Anchor to stream start: FIFO waiting must not accumulate over hours.
            # Skip missed slots instead of queueing catch-up cycles.
            interval = self.cfg['diagnostics_interval_s']
            self.next_cycle = (math.floor(elapsed / interval) + 1) * interval
            self.info['cycles_started'] += 1
        scope, address, command, fields = self.pending.popleft()
        start = self.clock()
        record = {'host_utc': datetime.now(timezone.utc).isoformat(),
                  'host_elapsed_s': start - self.start, 'scope': scope,
                  'address': '' if address is None else address,
                  'product_no': self.modules[address]['product_no'] if address is not None else '',
                  'command': command}
        self.info['calls'] += 1
        try:
            # Raw getters need the runtime transport lock and timeout.
            method = (getattr(self.base, command).__get__(self.backend, type(self.backend))
                      if self.base is not None else getattr(self.backend, command))
            args = () if address is None else (address,)
            result = self.device._call_locked_with_timeout(
                method, self.cfg['io_timeout_s'], 'zero_check_' + command, *args)
            record.update(read_duration_s=self.clock() - start, status=result[0])
            if result[0] != self.device.NO_ERR:
                self.info['failed_calls'] += 1
                for field in fields:
                    self.writer.writerow(dict(record, field=field, error=f'{command}: status {result[0]}'))
                self.stream.flush()
                if getattr(self.device, '_transport_poisoned', False):
                    raise RuntimeError('DLL blocked during diagnostics.')
                # A missing optional diagnostic is not a missing current sample.
                # Retain its error/empty fields; recover only if the link probe fails.
                probe = self.device.get_state(timeout_s=self.cfg['io_timeout_s'])
                if probe[0] in (-10, -11, -12, -13) and self.recover_read is not None:
                    self.recover_read(probe[0], command + ': controller probe')
                else:
                    state, label = checked(self.device, 'controller after diagnostic', probe)
                    if (int(state, 16) if isinstance(state, str) else int(state)) != 0:
                        raise RuntimeError(f'Unexpected controller state after diagnostic: {state} ({label}).')
            else:
                if len(result) != len(fields) + 1:
                    raise RuntimeError(f'{command}: unexpected diagnostic response length.')
                for field, value in zip(fields, result[1:]):
                    error = ''
                    if isinstance(value, (float, np.floating)) and not math.isfinite(value):
                        value, error = '', 'nonfinite'
                        self.info['invalid_fields'] += 1
                    if isinstance(value, (dict, list, tuple)):
                        value = json.dumps(value, ensure_ascii=False)
                    self.writer.writerow(dict(record, field=field, value=value, error=error))
                if command == 'get_state':
                    state = int(result[1], 16) if isinstance(result[1], str) else int(result[1])
                    if state != 0:
                        raise RuntimeError(f'Unexpected controller state during diagnostics: {result[1:]}.')
        except (Exception, KeyboardInterrupt) as exc:
            self.writer.writerow(dict(record, read_duration_s=self.clock() - start,
                                      error=f'{type(exc).__name__}: {exc}'))
            raise
        finally:
            self.stream.flush()
        if not self.pending:
            self.info['cycles_completed'] += 1


@contextlib.contextmanager
def prevent_sleep(set_state=None):
    """Prevent automatic Windows sleep, not lid closure or manual sleep.

    Restore the calling thread state on exit; the display may sleep.
    """
    if set_state is None and os.name != 'nt':
        yield {'active': False, 'reason': 'non-Windows (tests only)'}
        return
    if set_state is None:
        set_state = ctypes.windll.kernel32.SetThreadExecutionState
        set_state.argtypes = [ctypes.c_uint32]
        set_state.restype = ctypes.c_uint32
    previous = set_state(0x80000000 | 0x00000001)  # ES_CONTINUOUS | ES_SYSTEM_REQUIRED
    if not previous:
        raise RuntimeError('Windows sleep prevention failed; acquisition not started.')
    try:
        yield {'active': True, 'mode': 'ES_CONTINUOUS | ES_SYSTEM_REQUIRED',
               'manual_sleep_prevented': False}
    finally:
        if not set_state(previous):
            raise RuntimeError('Windows sleep-state restoration failed.')


def run_campaign(device, output_dir, cfg, provenance=None, clock=time.monotonic, sleep=time.sleep):
    """Run each series separately; verify shutdown and close COM before each pause."""
    validate_protocol(cfg)
    if getattr(device, '_transport_poisoned', False) or getattr(device, 'connected', False):
        raise RuntimeError('Use a disconnected, unblocked device.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    start = clock()
    report = {'schema': 3, 'kind': 'campaign', 'status': 'running', 'error': None,
              'started_utc': datetime.now(timezone.utc).isoformat(), 'protocol': dict(cfg),
              'provenance': provenance or {}, 'series': [], 'pauses': [],
              'planned_acquisition_s': cfg['series_count'] * cfg['duration_s'],
              'planned_disconnected_pause_s': (cfg['series_count'] - 1) * cfg['pause_s'],
              'conformity': 'not_assessable_no_acceptance_limits'}
    save_report(output_dir, report)
    try:
        with prevent_sleep() as sleep_state:
            report['sleep_prevention'] = sleep_state
            for index in range(1, cfg['series_count'] + 1):
                directory = f'series_{index:02d}'
                entry = {'number': index, 'directory': directory, 'status': 'starting',
                         'started_elapsed_s': clock() - start}
                report['series'].append(entry)
                save_report(output_dir, report)
                print(f'\nSERIES {index}/{cfg["series_count"]}: {cfg["duration_s"]/60:g} min; range {cfg["range"]}', flush=True)
                child = run_zero_check(device, output_dir / directory, cfg,
                                       provenance=provenance, clock=clock, sleep=sleep)
                entry.update(status=child['status'], finished_elapsed_s=clock() - start,
                             error=child['error'], cleanup=child['cleanup'])
                acquisition = child.get('acquisition') or {}
                entry['acquisition_complete'] = acquisition.get('complete', False)
                entry['read_recoveries'] = acquisition.get('read_recoveries', [])
                if 'started_monotonic_s' in acquisition:
                    entry['stream_start_elapsed_s'] = acquisition['started_monotonic_s'] - start
                    entry['stream_end_elapsed_s'] = entry['stream_start_elapsed_s'] + acquisition['duration_s']
                save_report(output_dir, report)
                if (child['status'] != 'complete' or not child['cleanup']['shutdown_confirmed']
                        or getattr(device, 'connected', False) or getattr(device, '_transport_poisoned', False)):
                    report['status'] = 'interrupted' if child['status'] == 'interrupted' else 'failed'
                    detail = child['error'] or '; '.join(child['cleanup'].get('errors', [])) or 'Cleanup not confirmed.'
                    report['error'] = f'Series {index}: {detail}'
                    break  # Never reconnect automatically after a failure.
                if index < cfg['series_count']:
                    pause = {'after_series': index, 'started_elapsed_s': clock() - start,
                             'requested_s': cfg['pause_s'], 'complete': False, 'port_closed': True}
                    report['pauses'].append(pause)
                    save_report(output_dir, report)
                    print(f'Port closed; waiting {cfg["pause_s"]:g} s before reconnecting.', flush=True)
                    try:
                        deadline = clock() + cfg['pause_s']
                        while clock() < deadline:
                            sleep(min(1.0, deadline - clock()))
                        pause['complete'] = True
                    finally:
                        pause['finished_elapsed_s'] = clock() - start
                        save_report(output_dir, report)
            else:
                report['status'] = 'complete'
    except (Exception, KeyboardInterrupt) as exc:
        report['status'] = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
        report['error'] = f'{type(exc).__name__}: {exc}'
        if report['series'] and report['series'][-1]['status'] == 'starting':
            report['series'][-1].update(status=report['status'], error=report['error'], finished_elapsed_s=clock()-start)
        # Handle errors outside the inner loop without retrying an unconfirmed shutdown.
        if report['series'] and not report['series'][-1].get('cleanup') and getattr(device, 'connected', False):
            report['emergency_cleanup'] = restore_and_shutdown(device, {}, cfg['io_timeout_s'])
    finally:
        report['finished_utc'] = datetime.now(timezone.utc).isoformat()
        report['elapsed_s'] = clock() - start
        report['completed_series'] = sum(s['status'] == 'complete' for s in report['series'])
        report['recorded_series'] = sum(s.get('acquisition_complete', False) for s in report['series'])
        report['link_released'] = not (getattr(device, 'connected', False) or getattr(device, '_transport_poisoned', False))
        save_report(output_dir, report)
    print(f'\nCAMPAIGN: {report["status"]}; {report["recorded_series"]}/{cfg["series_count"]} series recorded, {report["completed_series"]} completed normally.')
    if report['error']:
        print('CAMPAIGN STOPPED:', report['error'])
    return report


def analyse_campaign(output_dir):
    """Analyse one series at a time to limit memory use."""
    output_dir = Path(output_dir)
    report = json.loads((output_dir / 'report.json').read_text(encoding='utf-8'))
    if report.get('kind') != 'campaign':
        raise ValueError('Select the campaign folder, not a single-series folder.')
    summary, bins, tails, diagnostics, windows = [], [], [], [], []
    for entry in report['series']:
        directory = output_dir / entry['directory']
        if not (directory / 'raw.csv').exists() or not (directory / 'report.json').exists():
            continue
        raw, _, child = analyse_run(directory)
        entry['selected_points'] = int(raw['selected'].sum())
        entry['telemetry'] = child.get('telemetry')
        entry['acquisition_complete'] = (child.get('acquisition') or {}).get('complete', False)
        entry['read_recoveries'] = (child.get('acquisition') or {}).get('read_recoveries', [])
        # Use the host clock across runs; device counters may reset on reconnection.
        origin = entry.get('stream_start_elapsed_s', entry['started_elapsed_s'])
        for key, target in [('summary', summary), ('time_bins', bins), ('tail_summary', tails),
                            ('baseline_windows', windows)]:
            for row in child.get(key, []):
                target.append(dict(row, series=entry['number'], series_status=child['status'],
                                   campaign_start_s=origin + row.get('start_s', row['host_start_s']),
                                   campaign_end_s=origin + row.get('end_s', row['host_end_s'])))
        path = directory / 'telemetry.csv'
        if path.exists():
            data = pd.read_csv(path, float_precision='round_trip')
            data['series'] = entry['number']
            data['campaign_elapsed_s'] = origin + data['host_elapsed_s']
            diagnostics.append(data)
    report.update(summary=summary, time_bins=bins, tail_summary=tails, baseline_windows=windows,
                  recorded_series=sum(s.get('acquisition_complete', False) for s in report['series']))
    save_report(output_dir, report)
    table = pd.DataFrame(summary)
    table.to_csv(output_dir / 'series_summary.csv', index=False)
    window_columns = ['series', 'P/N', 'address', 'range', 'start_min', 'end_min',
                      'N', 'mean_pA', 'std_fA', 'drift_fA_min']
    pd.DataFrame([dict(row, start_min=row['campaign_start_s']/60,
                       end_min=row['campaign_end_s']/60) for row in windows],
                 columns=window_columns).to_csv(output_dir / 'baseline_windows.csv', index=False)
    telemetry = pd.concat(diagnostics, ignore_index=True) if diagnostics else pd.DataFrame(columns=TELEMETRY_COLUMNS + ['series', 'campaign_elapsed_s'])
    return table, telemetry, report


def shade_gaps(ax, report):
    series = report['series']
    for entry in series:
        origin = entry.get('stream_start_elapsed_s', entry['started_elapsed_s'])
        for incident in entry.get('read_recoveries', []):
            end = incident.get('resumed_elapsed_s', incident['finished_elapsed_s'])
            ax.axvspan((origin + incident['start_elapsed_s']) / 3600,
                       (origin + end) / 3600, color='0.9', zorder=0)
    for left, right in zip(series, series[1:]):
        if 'stream_end_elapsed_s' in left and 'stream_start_elapsed_s' in right:
            ax.axvspan(left['stream_end_elapsed_s'] / 3600, right['stream_start_elapsed_s'] / 3600,
                       color='0.9', zorder=0)
    ax.set_xlim(0, max(report.get('elapsed_s', 0), 1) / 3600)
    ax.grid(alpha=.2)
    ax.ticklabel_format(axis='y', style='plain', useOffset=False)


def plot_campaign(report, telemetry):
    import matplotlib.pyplot as plt
    pns = sorted({row['P/N'] for row in report.get('summary', [])})
    titles = {}
    for pn in pns:
        addresses = sorted({int(row['address']) for row in report.get('summary', [])
                            if row['P/N'] == pn and row.get('address') is not None})
        label = ', '.join(map(str, addresses)) or 'unknown'
        field = 'addresses' if len(addresses) > 1 else 'address'
        titles[pn] = f'P/N {pn} · DMMR {field} {label}'
    partial = '' if report['status'] == 'complete' else ' — INCOMPLETE'
    current, axes = plt.subplots(4, 2, figsize=(12, 11), layout='constrained', sharex=True)
    for ax, pn in zip(axes.flat, pns):
        for entry in report['series']:
            rows = [r for r in report.get('time_bins', []) if r['P/N'] == pn and r['series'] == entry['number']]
            if not rows:
                continue
            x = [(r['campaign_start_s'] + r['campaign_end_s']) / 7200 for r in rows]
            ax.errorbar(x, [r['mean_pA'] for r in rows],
                        yerr=[np.nan if r['std_fA'] is None else r['std_fA']/1000 for r in rows],
                        fmt='.-', ms=3, lw=.8, elinewidth=.6, color='C0', ecolor=(.12, .47, .71, .2))
        ax.set_title(titles[pn], fontsize=10)
        shade_gaps(ax, report)
    for ax in axes.flat[len(pns):]:
        ax.set_visible(False)
    current.supylabel('Current (pA)')
    current.supxlabel('Elapsed time (h, host clock)')
    condition = 'open, unshielded inputs' if report['protocol']['input_condition'] == 'open_unshielded' else 'open, shielded inputs'
    current.suptitle(f'DMMR · fixed range {report["protocol"]["range"]} · {condition}{partial}\n'
                     'Minute means ± standard deviation · shaded: pause or read recovery', fontsize=11)

    temperatures, axes = plt.subplots(4, 2, figsize=(12, 11), layout='constrained', sharex=True)
    def trace(ax, frame, field, label, color):
        rows = frame.loc[frame['field'] == field].copy()
        rows['numeric'] = pd.to_numeric(rows['value'], errors='coerce')
        rows.loc[(rows['status'] != 0) | rows['error'].notna(), 'numeric'] = np.nan
        labelled = False
        for _, group in rows.groupby('series', sort=True):
            if not group['numeric'].notna().any():
                continue
            # Leave gaps at errors and between series.
            ax.plot(group['campaign_elapsed_s']/3600, group['numeric'], '.-',
                    ms=2, lw=.8, color=color, label=label if not labelled else None)
            labelled = True
    module_data = telemetry.loc[telemetry['scope'] == 'module']
    for ax, pn in zip(axes.flat, pns):
        frame = module_data.loc[pd.to_numeric(module_data['product_no'], errors='coerce') == pn]
        trace(ax, frame, 'temp_cpu_c', 'TempCPU', 'C0')
        trace(ax, frame, 'temp_cpui_c', 'TempCPUi', 'C1')
        ax.set_title(titles[pn], fontsize=10)
        shade_gaps(ax, report)
        if not ax.lines:
            ax.text(.5, .5, 'No valid temperature readings', transform=ax.transAxes, ha='center')
            ax.set_yticks([])
    for ax in axes.flat[len(pns):]:
        ax.set_visible(False)
    if pns and axes.flat[0].lines:
        axes.flat[0].legend(fontsize=8)
    temperatures.supylabel('Temperature (°C)')
    temperatures.supxlabel('Elapsed time (h, host clock)')
    temperatures.suptitle(f'DMMR module temperatures{partial}\nSensor names from CGC API', fontsize=11)

    controller, axes = plt.subplots(3, 1, figsize=(12, 8), layout='constrained', sharex=True)
    control = telemetry.loc[telemetry['scope'] == 'controller']
    base = telemetry.loc[telemetry['scope'] == 'base']
    trace(axes[0], control, 'temp_cpu_c', 'Controller CPU', 'C0')
    trace(axes[0], base, 'temperature_c', 'Chassis', 'C1')
    axes[0].set_ylabel('Temperature (°C)')
    for field, label, color in [('volt_12v_v', '12 V', 'C0'), ('volt_5v0_v', '5 V', 'C1'), ('volt_3v3_v', '3.3 V', 'C2')]:
        trace(axes[1], control, field, label, color)
    axes[1].set_ylabel('Supply voltage (V)')
    trace(axes[2], base, 'fan_rpm', 'Fan speed', 'C0')
    axes[2].set_ylabel('Fan speed (rpm)')
    for ax in axes:
        shade_gaps(ax, report)
        if ax.lines:
            ax.legend(fontsize=8)
        else:
            ax.text(.5, .5, 'No valid readings', transform=ax.transAxes, ha='center')
            ax.set_yticks([])
    controller.supxlabel('Elapsed time (h, host clock)')
    controller.suptitle(f'DMMR controller and chassis{partial}', fontsize=11)
    return {'current_vs_time': current, 'module_temperatures': temperatures,
            'controller_diagnostics': controller}


## Run

Progress prints each minute. Keep the whole campaign folder, including partial runs: `raw.csv`, `telemetry.csv`, `report.json`, `native_*.log` and `dmmr_protocol_*.jsonl*` in each `series_XX/`.


In [ ]:
if os.name != 'nt' or struct.calcsize('P') != 8:
    raise RuntimeError('Hardware acquisition requires Windows and 64-bit Python.')
if isinstance(COM_PORT, bool) or not isinstance(COM_PORT, int) or not 1 <= COM_PORT <= 255:
    raise ValueError('Set COM_PORT to the port number (1-255).')
validate_protocol(PROTOCOL)

previous = globals().get('_zero_device')
if previous is not None and (getattr(previous, '_transport_poisoned', False) or getattr(previous, 'connected', False)):
    raise RuntimeError('Previous connection not released. Check shutdown locally; do not open another connection.')

PLUGIN_DIR = find_plugin_dir(PLUGIN_DIR)
DMMR = load_driver(PLUGIN_DIR)
if not all(hasattr(DMMR, name) for name in ('ReadRecovery', 'CommandRecovery')):
    raise RuntimeError('Update the complete dmmr/ folder (including vendor/) together with this notebook.')
paths = ['vendor/runtime/dmmr/dmmr.py', 'vendor/runtime/dmmr/dmmr_base.py',
         'vendor/runtime/dmmr/read_recovery.py',
         'vendor/runtime/_driver_common.py', 'vendor/runtime/dmmr/vendor/x64/COM-DMMR-8.dll']
provenance = {'plugin_dir': str(PLUGIN_DIR), 'python': sys.version, 'com': COM_PORT, 'baud_rate': BAUD_RATE,
              'sha256': {name: hashlib.sha256((PLUGIN_DIR / name).read_bytes()).hexdigest() for name in paths}}
RUN_DIR = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S_%f') + '_' + PROTOCOL['input_condition'])
_zero_device = DMMR('dmmr_zero_check', com=COM_PORT, baudrate=BAUD_RATE,
                    process_backend=False, log_dir=OUTPUT_ROOT)
report = run_campaign(_zero_device, RUN_DIR, PROTOCOL, provenance=provenance)

## Results

Plots show minute means ± sample standard deviation, temperatures and diagnostics. Grey bands mark interruptions. The shared axis uses host time; raw device timestamps are retained.

Tables summarise each series and its final five minutes if fully acquired. `baseline_windows.csv` contains means and slopes for complete 30-minute windows, excluding stream restarts. No drift is fitted across timestamp segments. Standard deviation includes drift, not just noise. Check settling before using an offset; temperature correlation does not establish causation.

This tests zero behaviour, not gain or linearity. Conformity needs applicable CGC limits and a reference current.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

# To analyse saved data, set RUN_DIR to the campaign folder.
summary, telemetry, report = analyse_campaign(RUN_DIR)
print(f'Campaign: {report["status"]}; {report["recorded_series"]}/{report["protocol"]["series_count"]} series recorded; {report.get("completed_series", 0)} completed normally.')
if report.get('error'):
    print('WARNING:', report['error'])
checks = pd.DataFrame([{
    'Series': entry['number'], 'Status': entry['status'],
    'Current samples': entry.get('selected_points', 0),
    'OFF + port closed': (entry.get('cleanup') or {}).get('shutdown_confirmed', False),
    'Diagnostic cycles': (entry.get('telemetry') or {}).get('cycles_completed', 0),
    'Diagnostic calls': (entry.get('telemetry') or {}).get('calls', 0),
    'Failed diagnostic calls': (entry.get('telemetry') or {}).get('failed_calls', 0),
    'Invalid diagnostic fields': (entry.get('telemetry') or {}).get('invalid_fields', 0),
} for entry in report['series']])
display(checks)
if not checks.empty and (checks['Diagnostic cycles'] == 0).any():
    print('WARNING: a series has no complete diagnostic cycle. Check telemetry.csv.')
if summary.empty:
    print('No valid current data. Check the reports and native logs; certificate values are not measurements from this run.')
else:
    cols = ['series', 'P/N', 'address', 'range', 'N', 'mean_pA', 'std_fA', 'drift_fA_min']
    display(summary[cols])
    print('Final five minutes of fully acquired series (check stability):')
    tails = pd.DataFrame(report['tail_summary'])
    if not tails.empty:
        display(tails[cols])
    windows = pd.read_csv(Path(RUN_DIR) / 'baseline_windows.csv')
    if not windows.empty:
        print('Complete baseline windows; start/end are elapsed minutes from campaign start:')
        display(windows[['series', 'P/N', 'start_min', 'end_min', 'N', 'mean_pA', 'drift_fA_min']])
    figures = plot_campaign(report, telemetry)
    for name, figure in figures.items():
        figure.savefig(Path(RUN_DIR) / (name + '.png'), dpi=150)
        display(figure)
        plt.close(figure)
    print('Data: series_XX folders. Summary: series_summary.csv.')
